In [ ]:
import json
PATH_65 = "sr/"
PATH_24 = "sr/"
PATH_BH = "b/"
PATH_EL = "xm/"
bks = []
with open(f'{PATH_65}488.json', 'r') as f:
    jsn = json.load(f)
    book_attrib = 'usfm'
#    print(jsn['books'])
    bks = [b[book_attrib] for b in jsn['books'] if book_attrib in b]
print(bks)

jd_only=True
bkslist = ["genesis", "exodus", "leviticus", "numbers", "deuteronomy", "joshua", "judges", "ruth", "1_samuel", "2_samuel", "1_kings", "2_kings", "1_chronicles", "2_chronicles", "ezra", "nehemiah", "esther", "job", "psalms", "proverbs", "ecclesiastes", "songs", "isaiah", "jeremiah", "lamentations", "ezekiel", "daniel", "hosea", "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk", "zephaniah", "haggai", "zechariah", "malachi", "matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
chps= [int(i) for i in "50 40 27 36 34 24 21 4 31 24 22 25 29 36 10 13 10 42 150 31 12 8 66 52 5 48 12 14 3 9 1 4 7 3 3 3 2 14 4 28 16 24 21 28 16 16 13 6 6 4 4 5 3 6 4 3 1 13 5 5 3 5 1 1 1 22".split(" ")]
idxl = {}
names = {}
revnames = {}
nt_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
for i in range (len(bkslist)):
    if (bkslist[i] in nt_stubchapters if jd_only else True):
        #idxl[bkslist[i].replace("_", "")[:3].upper()] = chps[i]
        idxl[bks[i]] = chps[i]
        names[bks[i]] = bkslist[i]
        revnames[bkslist[i]] = bks[i]

estr = ""
sepend ="; "    
sep =" "
for kp in idxl: estr += f"{kp}{sep}{idxl[kp]}{sepend}"
print(estr)
estr = ""
for kp in names: estr += f"{kp}{sep}{names[kp]}{sepend}"
print(estr)
def rnm(nm):
    return revnames[nm] if nm in revnames else nm

In [ ]:
#import bs4
from bs4 import BeautifulSoup

CONF_REMOVE_VERSE_NUMS = True
GREEN = '\033[92m'
RED = '\033[91m'
DIFF_BLUE =  '\x1b[38;5;12m\x1b[48;15;7m'
RESET = '\033[0m'
result ={}
result24 ={}

def html_verse_get_text_array(verses):
    result = []
    for v in verses:
        result.extend(html_verse_get_text(v))
    return result

def startswithInclasses(l, what):
    for c in l:
        if(c.startswith(what)): return True
    return False

def versetextfilter(elt):
    #print(elt.attrs['class'])
    return (elt.name == 'span' and 
            elt.has_attr('class') and 
            startswithInclasses(elt.attrs['class'], 'ChapterContent_content') and 
            elt.text.strip() != '')

def html_verse_get_text(v):
    contentels =v.find_all(versetextfilter)
    text = ""
    for v in contentels:
        if(text !=""): text += " "
        #if(CONF_REMOVE_VERSE_NUMS):
        text += v.get_text().lstrip('0123456789')\
                                       .replace(',', '').replace('.', '').replace('!', '')\
                                       .replace('?', '').replace(':', '').replace('"', '')\
                                       .replace('\'', '')
    return [{"form": i} for i in text.split()] 
        

for current_book in idxl:
    for chapter in range(1, idxl[current_book]+1):
        with open(f"{PATH_24}24_{current_book}_{chapter}.html", 'r') as f24:
            with open(f"{PATH_65}/65_{current_book}_{chapter}.html", 'r') as f:
                result[f"{current_book}_{chapter}"] = []
                result24[f"{current_book}_{chapter}"] = []

                cntnt = f.read()
                cntnt24 = f24.read()
                soup = BeautifulSoup(cntnt, 'html.parser')
                
                verses = soup.find_all('span', 
                      attrs={'data-usfm': lambda x: x and x.startswith(f"{current_book}.{chapter}")})
                ##NOTOK  filter(verses, key=attrib={'data-usfm': lambda x: x and x.startswith(f"{current_book}.{chapter}.{i}")})
                #filter(lambda x: x.attrib.get('data-usfm', '').startswith(f"{current_book}.{chapter}.{i}"), verses)
                num = max([int(e.attrs["data-usfm"].split(".")[-1]) for e in verses])
                #num = max(verses, key=lambda e: int(e.attrib["data-usfm"].split(".")[-1])).attrib["data-usfm"].split(".")[-1]
                result[f"{current_book}_{chapter}"] = [
                    html_verse_get_text_array(filter(lambda x: x.attrs.get('data-usfm', '').startswith(f"{current_book}.{chapter}.{i}"), verses)) for i in range(1, num+1)
                ]


                soup24 = BeautifulSoup(cntnt24, 'html.parser')
                
                verses24 = soup24.find_all('span', 
                      attrs={'data-usfm': lambda x: x and x.startswith(f"{current_book}.{chapter}")})
                num24 = max([int(e.attrs["data-usfm"].split(".")[-1]) for e in verses24])
                result24[f"{current_book}_{chapter}"] = [
                    html_verse_get_text_array(filter(lambda x: x.attrs.get('data-usfm', '').startswith(f"{current_book}.{chapter}.{i}"), verses24)) for i in range(1, num24+1)
                ]
                COL = f"{RED}" if len(verses) != len(verses24) else f"{GREEN}"
                print(f"{COL}{current_book}_{chapter}{RESET}")
                if(len(verses) != len(verses24)):
                    print(f"{COL}{len(verses)}'\t'{len(verses24)}{RESET}")
                #assert(verses==25)
                #print(result)
                #break
            #break
#print(result)
#soup.get_text()
print("all fulfilled")
#for v in verses:
#    result[f"{current_book}_{chapter}"].append(sentence_tokens)
#print(len(verses))
#assert(verses==25)

In [ ]:

verses[0].attrs.get('data-usfm', '').startswith(f"{current_book}.{chapter}.{1}")

In [ ]:
print(result.
      keys())
print(rnm("matthew")+"_"+str(1))
print(result[rnm("matthew")+"_"+str(1)][0])
print(result24[rnm("matthew")+"_"+str(1)][0])
print(result24[rnm("matthew")+"_"+str(1)][0][0])

In [ ]:
GREEN = '\033[92m'
RED = '\033[91m'
DIFF_BLUE =  '\x1b[38;5;12m\x1b[48;15;7m'
RESET = '\033[0m'
def compareArraysPrint(a1, a2, a1_name, a2_name):
    tb = "\t"
    HADDIFFBADBADPRACTICE = False
    for current_book in idxl:
        print(f"{DIFF_BLUE}{current_book}{RESET}")
        print(f"{a1_name}{tb}{a2_name}")
        for chapter in range(1, idxl[current_book]+1):
            for i in range(max(len(a1[f"{current_book}_{chapter}"]), len(a1[f"{current_book}_{chapter}"]))):
                a1str = str(len(a1[f"{current_book}_{chapter}"][i])) if i< len(a1[f"{current_book}_{chapter}"]) else "--"
                a2str = str(len(a2[f"{current_book}_{chapter}"][i])) if i< len(a2[f"{current_book}_{chapter}"]) else "--"
                COL = f"{GREEN}" if i< len(a1[f"{current_book}_{chapter}"]) and i< len(a2[f"{current_book}_{chapter}"]) \
                and len(a1[f"{current_book}_{chapter}"][i]) == len(a2[f"{current_book}_{chapter}"][i]) \
                    else f"{RED}"
                if COL == f"{RED}": HADDIFFBADBADPRACTICE = True
                print(f"{COL}{a1str}{tb}{a2str}{RESET}")
    if not HADDIFFBADBADPRACTICE: print(f"{RED}IDENTICAL BOOKS!!!!{RESET}")
    else: print(f"{GREEN}Books had at least one difference / variation{RESET}")
    return
            
            
compareArraysPrint(result, result24, "1965", "2024")

In [ ]:
print(result.
      keys())
print(rnm("matthew")+"_"+str(1))
print(result[rnm("matthew")+"_"+str(1)])
print(result[rnm("matthew")+"_"+str(1)][0][0])
print(result24[rnm("matthew")+"_"+str(1)][0][0])

In [ ]:
def ta(s):
    return s
#print(f"x: {xml_sources[ta("hebrews_1")][0][0]}")
#print(f"h: {html_sources["hebrews_1"][0][0]}")
# ANSI color codes
GREEN = '\033[92m'
RED = '\033[91m'
#ESC[38:5:⟨n⟩m Select foreground color      where n is a number from the table below
#ESC[48:5:⟨n⟩m Select background color
RED_W =  '\x1b[38;5;88m\x1b[48;5;7m'
DIFF_BLUE =  '\x1b[38;5;12m\x1b[48;15;7m'
DIFF_RED =  '\x1b[38;5;196m\x1b[48;15;7m'
#default bg, dark red letter
#'\x1b[38;5;88m'
#blue light bg, white letter
#'\x1b[5;46;88m'
#green bg whitel etter
#'\x1b[5;46;42m'
#green bg oragneleter'\x1b[5;31;42m'
# #'\033[38;5;55'
YELLOW = '\033[33m'
RESET = '\033[0m'
COLOR = RESET

xml_sources = result
html_sources = result24

class Visualizer_Cosole():
    def __init__(self):
        self.similars ={
    "θ" : "Θ",
    "Θ" : "θ"
}
        pass
    
    def letters_same(self, l1, l2):
        return self.words_same(l1, l2)
    
    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim!= ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim!= ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2
    
    def getdiff_word_Visual(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False)-> tuple[bool, bool, str]:
        p1 =""
        hadRed = False
        allReds = True
        COLOR = ''
        if(self.words_same(xx,yh)):
            allReds = False
            if not showGreens: return (hadRed, allReds, '')
            #COLOR = GREEN
            #s1 = f"{xx}"
            #s2 = f"{yh}{RESET}"
            return (hadRed, allReds, self.w_colordiff_OK(v, w, xx, yh))
        else:
            
            minlen = min(len(xx), len(yh))
            s1=""
            s2=""
            for i in range(minlen):
                if(xx[i]==yh[i]):
                    allReds = False
                    s1+=f"{GREEN}{xx[i]}{RESET}"
                    s2+=f"{GREEN}{yh[i]}{RESET}"
                    continue
                else:
                    if(self.letters_similiar(xx[i], yh[i])):
                        allReds = False
                        s1+=f"{YELLOW}{xx[i]}{RESET}"
                        s2+=f"{YELLOW}{yh[i]}{RESET}"
                        continue
                    hadRed = True
                    s1+=f"{RED}{xx[i]}{RESET}"
                    s2+=f"{RED}{yh[i]}{RESET}"
            #print(f"minlen: {minlen}")
            if(len(xx)> minlen):
                s1+=f"{RED}{xx[minlen:]}{RESET}"
            if(len(yh)> minlen):
                s2+=f"{RED}{yh[minlen:]}{RESET}"
            if(True):
                if(hadRed):
                    COLOR = RED
                else:
                    if not showYellows: return (hadRed, allReds, '')
                    COLOR = YELLOW
        p1 = f"{COLOR}{v}:{w}{RESET}"
        return (hadRed, allReds, f"{p1}\n{s1}\n{s2}")
    
    def w_colordiff_OK(self, v, w, xx: str, yh: str)-> str:
        COLOR = GREEN
        s1 = f"{xx}"
        s2 = f"{yh}{RESET}"
        p1 = f"{COLOR}{v}:{w}{RESET}"
        return f"{p1}\n{s1}\n{s2}"
            


seperator= ' '
dw = Visualizer_Cosole()
book = rnm("hebrews")
ch_nr = 1
chapt = book+"_"+str(ch_nr)
#v = 0
xm_c_len = len(xml_sources[ta(chapt)])
hm_c_len = len(html_sources[chapt])

minverses = min(xm_c_len, hm_c_len)
for v in range(minverses):
    w = 0
    xm_v_len = len(xml_sources[ta(chapt)][v])
    hm_v_len = len(html_sources[chapt][v])
    print(f"~~~~~~b:___ {book[:3].upper()} {ch_nr}:{v+1} ___:d~~~~~~")
    shorter_x = False
    if(xm_v_len!= hm_v_len):
        if(xm_v_len<hm_v_len): shorter_x = True
        print(f"{RED_W}xml: {xm_v_len} w\nhml: {hm_v_len} w ====== ({float(xm_v_len-hm_v_len)/float(xm_v_len+hm_v_len)*float(100):.1f}% )\n{RESET}~~~~~")
    else:
        print(f"{GREEN}wrd: {xm_v_len}\n{RESET}~~~~~")
    minwords = min (xm_v_len, hm_v_len)
    for i in range (minwords):
        w = i
        xx = xml_sources[ta(chapt)][v][w]["form"]
        yh = html_sources[chapt][v][w]["form"]
        hR, aR, resultGUI = dw.getdiff_word_Visual(v+1, w+1, xx, yh, showYellows=True)
        if(resultGUI):
            print(resultGUI)
    if(shorter_x):
        strbuild =''
        for i in range(xm_v_len, hm_v_len):
            strbuild += f"{html_sources[chapt][v][i]['form']}{seperator}"
        print(f"{DIFF_RED}+{strbuild}{RESET}")
    else : 
        if (hm_v_len < xm_v_len):
            strbuild =''
            
            for i in range(hm_v_len, xm_v_len):
                strbuild += f"{xml_sources[ta(chapt)][v][i]['form']}{seperator}"
            print(f"{DIFF_BLUE}-{strbuild}{RESET}")
# write a function to visualize the differences in the console                      
            
        


In [50]:
class Visualizer_HTML:
    def __init__(self):
        self.similars = {
            "θ" : "Θ",
            "Θ" : "θ"
        }
        self.html = []
        self.html.append("""
        <!DOCTYPE html>
        <html>
        <head>
            <style>
                .diff-table { border-collapse: collapse; width: 100%; }
                .diff-row { height: 1.5em; }
                .common { background-color: #e6ffe6; }
                .diff { background-color: #ffe6e6; }
                .similar { background-color: #fff7e6; }
                .header { background-color: #f0f0f0; }
            </style>
        </head>
        <body>
        """)

    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim != ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim != ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2

    def getdiff_word_html(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False) -> str:
        if self.words_same(xx, yh):
            if not showGreens: return ''
            return f'<tr class="diff-row"><td class="common">{xx}</td><td class="common">{yh}</td></tr>'
        
        minlen = min(len(xx), len(yh))
        s1 = s2 = ''
        for i in range(minlen):
            if xx[i] == yh[i]:
                s1 += f'<span class="common">{xx[i]}</span>'
                s2 += f'<span class="common">{yh[i]}</span>'
                continue
            elif self.letters_similiar(xx[i], yh[i]):
                s1 += f'<span class="similar">{xx[i]}</span>'
                s2 += f'<span class="similar">{yh[i]}</span>'
                continue
            s1 += f'<span class="diff">{xx[i]}</span>'
            s2 += f'<span class="diff">{yh[i]}</span>'
        
        if len(xx) > minlen:
            s1 += f'<span class="diff">{xx[minlen:]}</span>'
        if len(yh) > minlen:
            s2 += f'<span class="diff">{yh[minlen:]}</span>'
        
        return f'<tr class="diff-row"><td>{s1}</td><td>{s2}</td></tr>'

    def generate_html(self, xml_sources, html_sources, book, chapter):
        self.html.append(f'<h2>{book.upper()} {chapter}</h2>')
        chapt = f"{book}_{chapter}"
        xm_c_len = len(xml_sources[chapt])
        hm_c_len = len(html_sources[chapt])
        minverses = min(xm_c_len, hm_c_len)
        
        self.html.append('<table class="diff-table">')
        
        for v in range(minverses):
            xm_v_len = len(xml_sources[chapt][v])
            hm_v_len = len(html_sources[chapt][v])
            minwords = min(xm_v_len, hm_v_len)
            
            self.html.append(f'<tr class="header"><td colspan="2">Verse {v+1}</td></tr>')
            
            for i in range(minwords):
                xx = xml_sources[chapt][v][i]["form"]
                yh = html_sources[chapt][v][i]["form"]
                self.html.append(self.getdiff_word_html(v+1, i+1, xx, yh, showGreens=True, showYellows=True))
            
            if xm_v_len != hm_v_len:
                if xm_v_len < hm_v_len:
                    strbuild = ' '.join(f"{html_sources[chapt][v][i]['form']}" for i in range(xm_v_len, hm_v_len))
                    self.html.append(f'<tr><td></td><td class="diff">+{strbuild}</td></tr>')
                else:
                    strbuild = ' '.join(f"{xml_sources[chapt][v][i]['form']}" for i in range(hm_v_len, xm_v_len))
                    self.html.append(f'<tr><td>-{strbuild}</td><td></td></tr>')
        
        self.html.append('</table></body></html>')
        return '\n'.join(self.html)
    
    


In [51]:

dw = Visualizer_HTML()
book = rnm("hebrews")
chapt = 1

hmm = dw.generate_html(xml_sources, html_sources, book, chapt)
with open(f"hm1_{book}_{chapt}.html", 'w') as f:
    f.write(hmm)
        


In [91]:
GREEN = '<span class="green">'
RED = '<span class="red">'
YELLOW = '<span class="yellow">'
RESET = '</span>'
DIFF_RED = '<span class="diffred">'
DIFF_BLUE = '<span class="diffblue">'
class Visualizer_HTMLD(Visualizer_HTML):
    def __init__(self):
        self.similars = {
            "θ" : "Θ",
            "Θ" : "θ"
        }
        self.html = []
        self.html.append("""
        <!DOCTYPE html>
        <html>
        <head>
            <style>
                .diff-container { display: flex; flex-direction: column; gap: 1em; }
                .verse-row-header { display: flex; align-items: center; 
                background-color: #f0f0f0; }
                .common { background-color: #e6ffe6; }
                .diffx { background-color: #ffe6aa; }
                //.diffh { background-color: #ffe6e6; }
                .diffh {background-color: #ffe6aa;}
                .similar { background-color: #fff7e6; }
                .header { background-color: #f0f0f0; }
                .verse-row {
    display: flex;
    // flex-direction: column;
    align-items: flex-start;
    gap: 0.5em;
    min-height: 50px;
    padding: 0.25em;
}

.diffx {
    align-self: flex-start;
    background-color: #ffe6e6;
}

.diffh {
    align-self: flex-end;
    background-color: #fff7e6;//#e6ffe6;
}

.green {
    background-color: #e6ffe6;//#fff7e6;
    }

.yellow {
    background-color: #ffe6aa;
    }

.red {
    //background-color: #e6ffe6;//#fff7e6;
    }    

.common {
    background-color: #e6ffe6;//#fff7e6;
    align-self: anchor-center;
}

.diffred{
    background-color: #ff3333;
    align-self: anchor-center;
}
.diffblue{
    background-color: #3333ff;
    align-self: anchor-center;
}

/* Optional: responsive design */
@media (max-width: 768px) {
    .verse-row {
        gap: 0.25em;
        min-height: 40px;
    }
}
            </style>
        </head>
        <body>
        """)

    
    def letters_same(self, l1, l2):
        return self.words_same(l1, l2)
    
    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim!= ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim!= ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2
    
    def getdiff_word_VisualHTM(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False)-> str:
        p1 =""
        hadRed = False
        allReds = True
        COLOR = ''
        if(self.words_same(xx,yh)):
            allReds = False
            if not showGreens: return ''
            return self.w_colordiff_OK(v, w, xx, yh)
        else:
            
            minlen = min(len(xx), len(yh))
            s1=""
            s2=""
            for i in range(minlen):
                if(xx[i]==yh[i]):
                    allReds = False
                    s1+=f"{GREEN}{xx[i]}{RESET}"
                    s2+=f"{GREEN}{yh[i]}{RESET}"
                    continue
                else:
                    if(self.letters_similiar(xx[i], yh[i])):
                        allReds = False
                        s1+=f"{YELLOW}{xx[i]}{RESET}"
                        s2+=f"{YELLOW}{yh[i]}{RESET}"
                        continue
                    hadRed = True
                    s1+=f"{RED}{xx[i]}{RESET}"
                    s2+=f"{RED}{yh[i]}{RESET}"
            #print(f"minlen: {minlen}")
            if(len(xx)> minlen):
                s1+=f"{RED}{xx[minlen:]}{RESET}"
            if(len(yh)> minlen):
                s2+=f"{RED}{yh[minlen:]}{RESET}"
            if(True):
                if(hadRed):
                    COLOR = RED
                else:
                    if not showYellows: return ''
                    COLOR = YELLOW
        return f"{v}{s1}{RESET}\n{w}{s2}{RESET}"
    
    def w_colordiff_OK(self, v, w, xx: str, yh: str)-> str:
        COLOR = GREEN
        s1 = f"{xx}"
        s2 = f"{yh}"
        return f"{COLOR}{s1}\n{s2}{RESET}"


    def generate_html(self, xml_sources, html_sources, book, chapter):
        self.html.append(f'<h2>{book.upper()} {chapter}</h2>')
        chapt = f"{book}_{chapter}"
        xm_c_len = len(xml_sources[chapt])
        hm_c_len = len(html_sources[chapt])
        minverses = min(xm_c_len, hm_c_len)

        for v in range(len(xml_sources[chapt])):
            xm_v_len = len(xml_sources[chapt][v])
            hm_v_len = len(html_sources[chapt][v])
            minwords = min(xm_v_len, hm_v_len)
            self.html.append(f'<div class="verse-row-header">Verse {v+1}</div>')
            
            verse_html = []
            for w in range (minwords):
                xx = xml_sources[chapt][v][w]["form"]
                yh = html_sources[chapt][v][w]["form"]
                if xx == yh:
                    verse_html.append(f'<span class="common">{xx}</span>')
                else:
                    verse_html.append(f'<span class="diff-container">')
                    verse_html.append(self.getdiff_word_VisualHTM('<span class="diffx">', '<span class="diffh">', xx, yh, showGreens=True, showYellows=True))
                    verse_html.append(f'</span>')
            seperator = " "
            if(xm_v_len < hm_v_len):
                strbuild =''
                for i in range(xm_v_len, hm_v_len):
                    #strbuild += f"{html_sources[chapt][v][i]['form']}{seperator}"
                    strbuild += f"{DIFF_RED}{html_sources[chapt][v][i]['form']}{seperator}{RESET}"
                #verse_html.append(f"{DIFF_RED}+{strbuild}{RESET}")
                verse_html.append(f'<span class="diffh">{DIFF_RED}+{RESET}</span>{strbuild}')
            else : 
                if (hm_v_len < xm_v_len):
                    strbuild =''
                    for i in range(hm_v_len, xm_v_len):
                        #strbuild += f"{xml_sources[ta(chapt)][v][i]['form']}{seperator}"
                        strbuild += f"{DIFF_BLUE}{xml_sources[ta(chapt)][v][i]['form']}{seperator}{RESET}"
                    #verse_html.append(f"{DIFF_BLUE}-{strbuild}{RESET}")
                    verse_html.append(f'<span class="diffx">{DIFF_BLUE}-{RESET}</span>{strbuild}')
            
            self.html.append(f'<div class="verse-row">{" ".join(verse_html)}</div>')
        
        self.html.append('</body></html>')
        return '\n'.join(self.html)

In [92]:
dw = Visualizer_HTMLD()
book = rnm("hebrews")
chapt = 1

hmm = dw.generate_html(xml_sources, html_sources, book, chapt)
with open(f"hm1d_{book}_{chapt}.html", 'w') as f:
    f.write(hmm)

In [103]:
from datetime import datetime
from pathlib import Path
import os
date_str = datetime.now().strftime("%d-%m-%y")
dir_path = f"{date_str}-result"
Path(dir_path).mkdir(parents=True, exist_ok=True)
for current_book in idxl:
    print(names[current_book])
    Path(dir_path+os.path.sep+names[current_book]).mkdir(parents=True, exist_ok=True)
    for chapter in range(1, idxl[current_book]+1):
        print(names[current_book]+" "+str(chapter))
        with open(dir_path+os.path.sep+names[current_book]+os.path.sep+str(chapter)+'.html', 'w') as f:
            f.write(dw.generate_html(xml_sources, html_sources, current_book, chapter))

In [101]:
for b in idxl:
    print(b)
    print(idxl[b])
    print(names[b])

MAT
28
matthew
MRK
16
mark
LUK
24
luke
JHN
21
john
ACT
28
acts
ROM
16
romans
1CO
16
1_corinthians
2CO
13
2_corinthians
GAL
6
galatians
EPH
6
ephesians
PHP
4
philippians
COL
4
colossians
1TH
5
1_thessalonians
2TH
3
2_thessalonians
1TI
6
1_timothy
2TI
4
2_timothy
TIT
3
titus
PHM
1
philemon
HEB
13
hebrews
JAS
5
james
1PE
5
1_peter
2PE
3
2_peter
1JN
5
1_john
2JN
1
2_john
3JN
1
3_john
JUD
1
jude
REV
22
revelation
